# 🎬 IndoClip Pro — Google Colab GPU Backend (GitHub Edition)

Jalankan backend **IndoClip Pro** langsung dari repository GitHub Anda di Google Colab untuk memanfaatkan **NVIDIA T4 GPU (16GB VRAM)** dan internet datacenter gratis!

### 📌 Langkah Persiapan di Colab:
1. Di menu atas Colab, klik **Runtime** -> **Change runtime type** -> pilih **T4 GPU** -> klik **Save**.
2. Jalankan Sel 1 sampai Sel 5 secara berurutan.

### Sel 1: Verifikasi GPU NVIDIA

In [ ]:
!nvidia-smi

### Sel 2: Clone / Update Repository GitHub IndoClip

In [ ]:
import os, sys, shutil

GITHUB_REPO_URL = "https://github.com/nausamah27/IndoClip.git" # @param {type:"string"}
target_dir = "/content/IndoClip"

# Jika folder belum ada atau file main.py hilang, clone ulang secara bersih
if not os.path.exists(f"{target_dir}/backend/app/main.py"):
    print(f"Sedang meng-clone repository bersih dari {GITHUB_REPO_URL}...") 
    if os.path.exists(target_dir):
        shutil.rmtree(target_dir)
    !git clone {GITHUB_REPO_URL} {target_dir}
else:
    print("Repository sudah ada, menarik update terbaru dari GitHub...")
    !cd {target_dir} && git pull origin main

backend_dir = f"{target_dir}/backend"
os.chdir(backend_dir)
if backend_dir not in sys.path:
    sys.path.insert(0, backend_dir)
os.environ["PYTHONPATH"] = backend_dir
print(f"\n✅ Berhasil masuk ke direktori backend: {os.getcwd()}")

### Sel 3: Install Dependensi Python untuk GPU & FFmpeg

In [ ]:
!apt-get update -qq && apt-get install -y -qq ffmpeg
!pip install -q fastapi "uvicorn[standard]" yt-dlp faster-whisper google-genai opencv-python numpy scipy python-dotenv sse-starlette aiofiles requests

### Sel 4: Download Cloudflare Tunnel (Tunneling Publik Gratis)

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

### Sel 5: Jalankan Backend IndoClip Pro & Dapatkan URL Publik HTTPS

In [ ]:
import os, sys, subprocess, time, re, requests, shutil, site

backend_dir = "/content/IndoClip/backend"
if not os.path.exists(f"{backend_dir}/app/main.py"):
    raise FileNotFoundError(f"Folder backend tidak ditemukan di {backend_dir}! Pastikan Sel 2 sudah dijalankan.")

os.chdir(backend_dir)
if backend_dir not in sys.path:
    sys.path.insert(0, backend_dir)
os.environ["PYTHONPATH"] = backend_dir

# 0. Verifikasi dependensi & CUDA cuBLAS 12 (Otomatis install jika terlewat)
try:
    import yt_dlp, faster_whisper, cv2
except ImportError:
    print("⏳ Dependensi belum lengkap, menginstall otomatis (hanya butuh ~1 menit)...\n")
    !apt-get update -qq && apt-get install -y -qq ffmpeg
    !pip install -q fastapi "uvicorn[standard]" yt-dlp faster-whisper google-genai opencv-python numpy scipy python-dotenv sse-starlette aiofiles requests nvidia-cublas-cu12 nvidia-cudnn-cu12

# Pastikan Cloudflared sudah terpasang
if not shutil.which("cloudflared"):
    print("⏳ Mengunduh Cloudflared Tunnel...\n")
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
    !dpkg -i cloudflared-linux-amd64.deb

# Siapkan LD_LIBRARY_PATH untuk CUDA GPU cuBLAS 12
extra_ld_paths = []
for p in site.getsitepackages():
    for sub in ["cublas", "cudnn"]:
        target = os.path.join(p, "nvidia", sub, "lib")
        if os.path.isdir(target):
            extra_ld_paths.append(target)
backend_env = os.environ.copy()
if extra_ld_paths:
    backend_env["LD_LIBRARY_PATH"] = ":".join(extra_ld_paths) + ":" + backend_env.get("LD_LIBRARY_PATH", "")

# 1. Tes import langsung di Python untuk memastikan tidak ada error library
print("⏳ [1/4] Menguji modul backend FastAPI...")
try:
    from app.main import app
    print("✅ Modul backend berhasil dimuat tanpa error!")
except Exception as e:
    print(f"❌ Gagal memuat app.main: {e}")
    raise e

# 2. Hentikan proses lama jika ada yang masih memakai port 8000
!fuser -k 8000/tcp 2>/dev/null || true
!killall -9 cloudflared 2>/dev/null || true

# 3. Jalankan Uvicorn dengan file log & CUDA library path
print("⏳ [2/4] Menjalankan server Uvicorn pada port 8000...")
log_file = open("/content/uvicorn.log", "w")
backend_proc = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"],
    cwd=backend_dir,
    env=backend_env,
    stdout=log_file,
    stderr=subprocess.STDOUT
)

# 4. Tunggu hingga backend benar-benar online & tes /api/health
print("⏳ [3/4] Menunggu backend online di http://127.0.0.1:8000 ...")
online = False
for attempt in range(20):
    time.sleep(1)
    if backend_proc.poll() is not None:
        log_file.close()
        with open("/content/uvicorn.log", "r") as f:
            print("❌ Backend Uvicorn berhenti tak terduga! Log error:")
            print(f.read())
        raise RuntimeError("Backend gagal start.")
    try:
        r = requests.get("http://127.0.0.1:8000/api/health", timeout=2)
        if r.status_code == 200:
            online = True
            data = r.json()
            print(f"✅ Backend Online! Status: {data.get('status')} | Komputasi: {data.get('cuda_device', 'cpu').upper()} ({data.get('compute_type')})")
            break
    except Exception:
        pass

if not online:
    log_file.close()
    with open("/content/uvicorn.log", "r") as f:
        print("❌ Backend tidak merespons healthcheck dalam 20 detik! Log:")
        print(f.read())
    raise TimeoutError("Backend healthcheck timeout.")

# 5. Jalankan Cloudflare Tunnel
print("⏳ [4/4] Mengaktifkan Cloudflare Tunnel...")
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stderr=subprocess.PIPE,
    text=True
)

tunnel_url = None
while True:
    line = tunnel_proc.stderr.readline()
    if "trycloudflare.com" in line:
        match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
        if match:
            tunnel_url = match.group(0)
            print("\n=======================================================")
            print(f"🚀 URL BACKEND ANDA TELAH AKTIF & TERVERIFIKASI:")
            print(f"👉 {tunnel_url}")
            print("=======================================================")
            print("1. Buka website Vercel Anda di browser.")
            print("2. Klik tombol 'Backend: Offline / Set URL' di pojok kanan atas.")
            print(f"3. Tempelkan URL di atas ({tunnel_url}) dan klik 'Uji & Simpan URL'.")
            print("=======================================================\n")
            break

try:
    backend_proc.wait()
except KeyboardInterrupt:
    print("Backend dihentikan oleh user.")
    backend_proc.kill()
    tunnel_proc.kill()
